# Anima Text Encoder Bridge Calibration (JupyterLab)

This notebook calibrates an alternate source encoder (default target: Qwen3.5-0.8B-Base) into the representation space expected by Anima's Qwen3-0.6B-trained adapter.

Calibration is gradient-free. It does **not** fine-tune either encoder.

Before running, apply the Qwen3.5 Bridge/LongSource patch and Encoder Profile v2 incremental patch. Corpus construction rules are in `docs/calibration_prompt_corpus.md`.


In [ ]:
from pathlib import Path
import sys

# Resolve the repository whether this notebook is launched from repo root or notebooks/.
_here = Path.cwd().resolve()
if (_here / "scripts" / "calibrate_text_encoder_bridge.py").exists():
    REPO_ROOT = _here
elif (_here.parent / "scripts" / "calibrate_text_encoder_bridge.py").exists():
    REPO_ROOT = _here.parent
else:
    raise RuntimeError("Run this notebook from the diffusers-anima repository or its notebooks/ directory.")

sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "scripts"))
print("REPO_ROOT =", REPO_ROOT)


## Configuration

`PROMPTS = None` uses the deterministic built-in corpus. Set it to a UTF-8 text file for a custom corpus. Start with `BUNDLE_SOURCE_WEIGHTS = False`; after the bridge is validated, enable it to create a single aligned-encoder artifact.


In [ ]:
SOURCE_MODEL = "/workspace/qwen3.5-0.8b-base.safetensors"
REFERENCE_MODEL = "/workspace/qwen3-0.6b-base.safetensors"
OUTPUT = "/workspace/qwen35_08b_to_qwen3_06b.profile.safetensors"

SOURCE_TOKENIZER = "Qwen/Qwen3.5-0.8B-Base"
REFERENCE_TOKENIZER = "Qwen/Qwen3-0.6B-Base"

# None -> built-in deterministic 4096-line corpus.
PROMPTS = None
# Optional: save the exact generated/loaded base corpus for reproducibility.
DUMP_PROMPTS = "/workspace/anima_bridge_prompts_used.txt"

DEVICE = "auto"          # auto / cuda / mps / cpu
BATCH_SIZE = 8            # raise if VRAM allows
MAX_LENGTH = 2048
POOLING = "both"          # mean / last / blend / both
LAST_WEIGHT = 0.65        # only used by pooling="blend"
INCLUDE_PHRASES = True
MIN_PHRASE_CHARS = 3
VALIDATION_FRACTION = 0.08
SEED = 3571
DEFAULT_PROMPT_COUNT = 4096
BUNDLE_SOURCE_WEIGHTS = False


In [ ]:
from calibrate_text_encoder_bridge import (
    BridgeCalibrationConfig,
    calibrate_text_encoder_bridge,
    load_calibration_prompts,
    make_jupyter_progress_callback,
    preview_calibration_corpus,
)

report = preview_calibration_corpus(
    PROMPTS,
    default_count=DEFAULT_PROMPT_COUNT,
    seed=SEED,
    include_phrases=INCLUDE_PHRASES,
    min_phrase_chars=MIN_PHRASE_CHARS,
)
for key, value in report.items():
    if key != "warnings":
        print(f"{key:24s}: {value}")
if report["warnings"]:
    print("\nWarnings:")
    for warning in report["warnings"]:
        print(" -", warning)


## Optional corpus inspection

Inspect a few examples before model loading. With a custom corpus, this is also a quick way to verify comments/blank lines are being ignored as intended.


In [ ]:
base_prompts, corpus_source = load_calibration_prompts(
    PROMPTS, default_count=DEFAULT_PROMPT_COUNT, seed=SEED
)
print("source:", corpus_source)
print("lines :", len(base_prompts))
print()
for i, prompt in enumerate(base_prompts[:12]):
    print(f"[{i:02d}] {prompt}")


## Run calibration

The progress display updates in one output area and does not require `ipywidgets`. On CUDA, start with batch size 8 if you are unsure about VRAM.


In [ ]:
config = BridgeCalibrationConfig(
    source_model=SOURCE_MODEL,
    reference_model=REFERENCE_MODEL,
    output=OUTPUT,
    source_tokenizer=SOURCE_TOKENIZER,
    reference_tokenizer=REFERENCE_TOKENIZER,
    prompts=PROMPTS,
    dump_prompts=DUMP_PROMPTS,
    default_prompt_count=DEFAULT_PROMPT_COUNT,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    pooling=POOLING,
    last_weight=LAST_WEIGHT,
    max_length=MAX_LENGTH,
    include_phrases=INCLUDE_PHRASES,
    min_phrase_chars=MIN_PHRASE_CHARS,
    validation_fraction=VALIDATION_FRACTION,
    seed=SEED,
    bundle_source_weights=BUNDLE_SOURCE_WEIGHTS,
    cleanup_models=True,
)

progress = make_jupyter_progress_callback()
result = calibrate_text_encoder_bridge(config, progress_callback=progress)
print("\n" + result.summary())


## Result metadata

The recommended center/variance strengths and validation metrics are embedded in the profile. Runtime loading can use those recommendations automatically.


In [ ]:
for key in (
    "format",
    "artifact_kind",
    "source_family",
    "target_family",
    "corpus_source",
    "calibration_corpus_sha256",
    "corpus_lines",
    "expanded_anchors",
    "validation_samples",
    "validation_cosine_before",
    "validation_cosine_after",
    "validation_rmse_after",
    "recommended_center_strength",
    "recommended_variance_strength",
):
    print(f"{key:34s}: {result.metadata.get(key)}")


## Next step

For the first image tests, keep `BUNDLE_SOURCE_WEIGHTS=False` and load the normal Qwen3.5-0.8B encoder plus this profile. Once the bridge is validated, rerun with `BUNDLE_SOURCE_WEIGHTS=True` to create a single `aligned_encoder` file that can replace the standalone 0.8B encoder path.


## Optional: package as final v3 encoder
After validating the v2 bridge, package the unchanged Qwen3.5 backbone + one global Anima conditioning head into a single v3 encoder. This is non-destructive and can preserve extra source semantics through bounded expansion slots.


In [ ]:
FINALIZE_TO_V3 = False
FINAL_ENCODER_OUTPUT = '/workspace/Anima-Qwen3.5-0.8B-Final.safetensors'
SEMANTIC_EXPANSION_STRENGTH = 0.25
SEMANTIC_EXPANSION_MAX_TOKENS = 16


In [ ]:
if FINALIZE_TO_V3:
    from finalize_anima_text_encoder import FinalEncoderConfig, finalize_anima_text_encoder
    final = finalize_anima_text_encoder(FinalEncoderConfig(
        bridge_profile=OUTPUT,
        source_model=SOURCE_MODEL,
        output=FINAL_ENCODER_OUTPUT,
        semantic_expansion_strength=SEMANTIC_EXPANSION_STRENGTH,
        semantic_expansion_max_tokens=SEMANTIC_EXPANSION_MAX_TOKENS,
    ))
    print(final.summary())
